# RAG Document Q&A 

## 📦 Step 0 — Install Dependencies
Installs every third-party library the pipeline needs in one shot:
- **groq** — LLM client (Groq-hosted `openai/gpt-oss-20b`) for answer generation
- **chromadb** — persistent vector database used to store chunk embeddings
- **pypdf**, **python-docx** — document loaders for PDF and DOCX
- **streamlit**, **ipywidgets** — UI layers (notebook widgets + optional web UI)
- **rank-bm25**, **numpy** — supporting utilities
- **python-dotenv** — loads the `GROQ_API_KEY` from a local `.env` file
- **fastembed** — fast, local (CPU) embedding model (`BAAI/bge-small-en-v1.5`)

In [1]:
! pip install -q groq chromadb pypdf python-docx streamlit rank-bm25 numpy python-dotenv fastembed ipywidgets

## ⚙️ Step 1 — Imports & Configuration
Imports all libraries, loads environment variables via `python-dotenv`, and validates that `GROQ_API_KEY` is present (fails fast with a clear error if it's missing — the key is never hard-coded). Also defines the global configuration used throughout the notebook:
- `LLM_MODEL` — Groq model used for generation (`openai/gpt-oss-20b`)
- `EMBEDDING_MODEL` — FastEmbed model used for embeddings (`BAAI/bge-small-en-v1.5`)
- `CHUNK_SIZE` / `CHUNK_OVERLAP` — chunking parameters (500 chars / 100 overlap)
- `TOP_K` — number of chunks retrieved per query
- `DATA_DIR` / `CHROMA_DIR` / `COLLECTION_NAME` — folder for source documents, folder for the persistent Chroma index, and the Chroma collection name

In [2]:
import os
import re
import io
import json
import pickle
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any

from dotenv import load_dotenv

# Document loaders
from pypdf import PdfReader
from docx import Document as DocxDocument

# Vector DB
import chromadb
from chromadb.config import Settings

# Embeddings
from fastembed import TextEmbedding

# LLM
from groq import Groq

# ------------------------------------------------------------
# LOAD ENVIRONMENT VARIABLES
# ------------------------------------------------------------
load_dotenv(override=True)

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "❌ GROQ_API_KEY not found.\n"
        "Add it to your .env file like:\n"
        "GROQ_API_KEY=gsk_xxxxxxxxxxxxxxxxxxxx"
    )

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------
LLM_MODEL         = "openai/gpt-oss-20b"          # updated Groq model
EMBEDDING_MODEL   = "BAAI/bge-small-en-v1.5"      # fastembed model
CHUNK_SIZE        = 500
CHUNK_OVERLAP     = 100
TOP_K             = 5
DATA_DIR          = "data"
CHROMA_DIR        = "chroma_db"
COLLECTION_NAME   = "documents"

print("✅ Configuration loaded")
print(f"   • LLM model       : {LLM_MODEL}")
print(f"   • Embedding model : {EMBEDDING_MODEL}")
print(f"   • Key loaded      : {bool(GROQ_API_KEY)}")
print(f"   • Key preview     : {GROQ_API_KEY[:8]}...{GROQ_API_KEY[-4:]}")

c:\Users\pasal\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.6.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


✅ Configuration loaded
   • LLM model       : openai/gpt-oss-20b
   • Embedding model : BAAI/bge-small-en-v1.5
   • Key loaded      : True
   • Key preview     : gsk_BIYG...dAzO


## ✅ Step 2 — Sanity-Check the Groq API
Lists the models currently available on the Groq account and then sends a trivial "say hello" chat completion using `LLM_MODEL`, to confirm the API key and model name both work before building the rest of the pipeline on top of them.

In [3]:
# ============================================================
# TEST GROQ API + SHOW AVAILABLE MODELS
# ============================================================

client = Groq(api_key=GROQ_API_KEY)

print("Available Groq models:\n")
try:
    for m in client.models.list().data:
        print(" •", m.id)
except Exception as e:
    print("Could not list models:", e)

print("\n" + "=" * 60)

try:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful AI assistant."},
            {"role": "user",   "content": "Say hello and confirm Groq is working."}
        ],
        temperature=0.2,
        max_tokens=200,
    )
    print("✅ Groq API is working")
    print("Model   :", LLM_MODEL)
    print("Response:", response.choices[0].message.content.strip())

except Exception as e:
    print("❌ Groq API failed")
    print(type(e).__name__, "-", e)

Available Groq models:

 • meta-llama/llama-prompt-guard-2-22m
 • qwen/qwen3.8-27b
 • openai/gpt-oss-20b
 • openai/gpt-oss-120b
 • canopylabs/orpheus-v1-english
 • allam-2-7b
 • whisper-large-v3
 • openai/gpt-oss-safeguard-20b
 • meta-llama/llama-prompt-guard-2-86m
 • whisper-large-v3-turbo
 • canopylabs/orpheus-arabic-saudi

✅ Groq API is working
Model   : openai/gpt-oss-20b
Response: Hello! Groq is working.


## 📄 Step 3 — Document Ingestion
Loads the raw knowledge-base documents from `DATA_DIR` and extracts clean text:
- `clean_text()` strips standalone page numbers, collapses repeated blank lines/spaces
- `load_pdf()` reads a PDF page-by-page with `pypdf` and keeps per-page metadata
- `load_txt()` / `load_docx()` read plain-text and Word files as a single "page"
- `load_documents()` walks `DATA_DIR`, dispatches each file to the right loader based on its extension, and prints a line per file as it's loaded

The result, `documents`, is a flat list of `{text, source, page}` records — one per page/section across all 5 source PDFs.

In [4]:
def clean_text(text: str) -> str:
    """Strip excessive whitespace, standalone page numbers, etc."""
    text = re.sub(r"\n\s*\d+\s*\n", "\n", text)     # remove standalone page numbers
    text = re.sub(r"\n{3,}", "\n\n", text)          # collapse newlines
    text = re.sub(r"[ \t]{2,}", " ", text)          # collapse spaces
    return text.strip()


def load_pdf(file_path: str) -> List[Dict[str, Any]]:
    reader = PdfReader(file_path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = clean_text(page.extract_text() or "")
        if text.strip():
            pages.append({
                "text": text,
                "source": Path(file_path).name,
                "page": i
            })
    return pages


def load_txt(file_path: str) -> List[Dict[str, Any]]:
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        text = clean_text(f.read())
    return [{"text": text, "source": Path(file_path).name, "page": 1}]


def load_docx(file_path: str) -> List[Dict[str, Any]]:
    doc = DocxDocument(file_path)
    text = clean_text("\n".join(p.text for p in doc.paragraphs))
    return [{"text": text, "source": Path(file_path).name, "page": 1}]


def load_documents(folder: str) -> List[Dict[str, Any]]:
    docs = []
    folder_path = Path(folder)
    for file in sorted(folder_path.iterdir()):
        if file.suffix.lower() == ".pdf":
            docs.extend(load_pdf(str(file)))
            print(f"  📄 Loaded PDF : {file.name}")
        elif file.suffix.lower() == ".txt":
            docs.extend(load_txt(str(file)))
            print(f"  📄 Loaded TXT : {file.name}")
        elif file.suffix.lower() == ".docx":
            docs.extend(load_docx(str(file)))
            print(f"  📄 Loaded DOCX: {file.name}")
    return docs


# ---- Run ingestion ----
documents = load_documents(DATA_DIR)
print(f"\n✅ Total pages/sections loaded: {len(documents)}")
if documents:
    print(f"Sample: {documents[0]['source']} - page {documents[0]['page']}")

  📄 Loaded PDF : Artificial Intelligence.pdf
  📄 Loaded PDF : BUSINESS.pdf
  📄 Loaded PDF : LAW.pdf
  📄 Loaded PDF : SCIENCE.pdf
  📄 Loaded PDF : The MCU.pdf

✅ Total pages/sections loaded: 15
Sample: Artificial Intelligence.pdf - page 1


## ✂️ Step 4 — Text Chunking
Splits each loaded page into overlapping fixed-size chunks so no context is lost at a boundary:
- `chunk_text()` — sliding-window character chunking (`CHUNK_SIZE` chars, `CHUNK_OVERLAP` overlap)
- `build_chunks()` — applies chunking to every document and attaches metadata to each chunk: source filename, page number, and a unique `chunk_id` (`filename_pPAGE_cN`)

This produces the final `chunks` list that everything downstream (embedding, indexing, retrieval) is built on.

In [5]:
def chunk_text(text: str,
               chunk_size: int = CHUNK_SIZE,
               overlap: int = CHUNK_OVERLAP) -> List[str]:
    """Sliding-window character chunking with overlap."""
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= text_len:
            break
        start = end - overlap
    return chunks


def build_chunks(documents: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    all_chunks = []
    for doc in documents:
        for i, chunk in enumerate(chunk_text(doc["text"])):
            all_chunks.append({
                "text":     chunk,
                "source":   doc["source"],
                "page":     doc["page"],
                "chunk_id": f"{doc['source']}_p{doc['page']}_c{i}"
            })
    return all_chunks


chunks = build_chunks(documents)
print(f"✅ Created {len(chunks)} chunks")
if chunks:
    print(f"\nExample chunk:\n{chunks[0]['text'][:300]}...")

✅ Created 83 chunks

Example chunk:
"Artificial Intelligence: Foundations, 
Applications, and Ethics " 
1. Introduction 
Artificial Intelligence (AI) is a field of computer science focused on developing 
machines and software capable of performing tasks that normally require 
human intelligence. These tasks include learning, reasoning...


## 🧬 Step 5 — Embedding Model
Initializes the local FastEmbed model (`BAAI/bge-small-en-v1.5`) and wraps it in `embed_texts()`, a **batched** embedding helper (all texts are embedded in one call, never in a one-chunk-at-a-time loop). A quick smoke test embeds the first 5 chunks and prints the resulting embedding matrix shape to confirm the dimensionality (384-d vectors).

In [6]:
embedder = TextEmbedding(model_name=EMBEDDING_MODEL)

def embed_texts(texts: List[str]) -> np.ndarray:
    """Batched embedding via fastembed."""
    embeddings = list(embedder.embed(texts))
    return np.array(embeddings, dtype=np.float32)


sample_embeds = embed_texts([c["text"] for c in chunks[:5]])
print(f"✅ Embedding shape: {sample_embeds.shape}")

✅ Embedding shape: (5, 384)


## 🗄️ Step 6 — Vector Database (ChromaDB) — Indexing
Builds the persistent vector store, kept fully separate from the querying step below:
- `get_chroma_client()` — opens a `PersistentClient` pointed at `CHROMA_DIR`, so the index survives across notebook restarts and isn't rebuilt on every run
- `index_chunks()` — (re)creates the `documents` collection with cosine similarity, batch-embeds every chunk's text, and adds ids/embeddings/documents/metadata (`source`, `page`) to the collection in a single `collection.add(...)` call

Running this cell performs the actual indexing and reports the resulting collection size.

In [7]:
def get_chroma_client():
    return chromadb.PersistentClient(path=CHROMA_DIR)


def index_chunks(chunks: List[Dict[str, Any]], reset: bool = True):
    client = get_chroma_client()

    if reset:
        try:
            client.delete_collection(COLLECTION_NAME)
        except Exception:
            pass

    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )

    texts = [c["text"] for c in chunks]
    print(f"🔄 Embedding {len(texts)} chunks...")
    embeddings = embed_texts(texts)

    collection.add(
        ids         = [c["chunk_id"] for c in chunks],
        documents   = texts,
        embeddings  = embeddings.tolist(),
        metadatas   = [{"source": c["source"], "page": c["page"]} for c in chunks]
    )
    print(f"✅ Indexed {len(chunks)} chunks into ChromaDB")
    return collection


collection = index_chunks(chunks, reset=True)
print(f"📊 Collection count: {collection.count()}")

🔄 Embedding 83 chunks...
✅ Indexed 83 chunks into ChromaDB
📊 Collection count: 83


## 🔍 Step 7 — Retrieval
`retrieve()` is the **querying half** of the pipeline (kept separate from indexing above): it embeds the incoming question with the same FastEmbed model, runs a similarity search against the persisted Chroma collection, and returns the top-`k` chunks together with their source filename, page number, and similarity distance — `k` is a configurable parameter (defaults to `TOP_K`).

In [8]:
def retrieve(query: str, top_k: int = TOP_K) -> List[Dict[str, Any]]:
    if not query or not query.strip():
        return []

    client = get_chroma_client()
    collection = client.get_collection(name=COLLECTION_NAME)

    q_emb = embed_texts([query])[0].tolist()

    results = collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    retrieved = []
    for i, doc in enumerate(documents):
        meta = metadatas[i] if i < len(metadatas) and metadatas[i] else {}
        dist = distances[i] if i < len(distances) else None
        retrieved.append({
            "text":     doc,
            "source":   meta.get("source", "Unknown"),
            "page":     meta.get("page", "Unknown"),
            "distance": dist,
        })
    return retrieved

## 📝 Step 8 — Prompt Construction
`build_prompt()` turns the retrieved chunks into a strict, grounded prompt for the LLM:
- If nothing was retrieved, it returns a prompt that forces the fixed fallback answer, so the model never hallucinates from its own training data
- Otherwise, it numbers each retrieved chunk with its source filename and page, wraps them in a `CONTEXT` block, and appends explicit rules — answer only from context, cite every claim as `[Source: filename, page X]`, and use the exact fallback sentence when the answer isn't present

In [9]:
def build_prompt(query: str, retrieved: List[Dict[str, Any]]) -> str:
    if not retrieved:
        return f"""
You are a document question-answering assistant.

The document database returned no relevant information.

User question:
{query}

Answer exactly:
I don't have enough information in the provided documents to answer this.
"""

    blocks = []
    for i, item in enumerate(retrieved, start=1):
        blocks.append(
            f"[{i}]\n"
            f"Source: {item.get('source','Unknown')}\n"
            f"Page: {item.get('page','Unknown')}\n\n"
            f"{item.get('text','')}\n"
        )

    context = "\n".join(blocks)

    return f"""
You are a grounded document Q&A assistant.

Your job is to answer the user's question using ONLY
the information contained in the provided context.

IMPORTANT RULES:

1. Do NOT use outside knowledge.
2. Do NOT invent information.
3. Do NOT make assumptions.
4. If the answer is not present in the context, respond exactly:

I don't have enough information in the provided documents to answer this.

5. Every factual claim must include a citation.
6. Citation format: [Source: filename, page X]
7. Keep the answer concise and factual.
8. If multiple sources support an answer, cite each relevant source.
9. Preserve important numbers, names, dates and technical terms.

--------------------------------------------------
CONTEXT
--------------------------------------------------

{context}

--------------------------------------------------
USER QUESTION
--------------------------------------------------

{query}

--------------------------------------------------
ANSWER
--------------------------------------------------
"""

## 🤖 Step 9 — Answer Generation (Full RAG Pipeline)
Creates the `groq_client` and defines `generate_answer()`, which chains everything together: **retrieve → build prompt → call Groq LLM → return grounded answer + sources**. If retrieval returns nothing, it short-circuits straight to the "not enough information" answer; if the Groq call itself fails, the exception is caught and surfaced as the answer text instead of crashing.

In [10]:
# ============================================================
# RAG PIPELINE
# ============================================================

groq_client = Groq(api_key=GROQ_API_KEY)


def generate_answer(query: str, top_k: int = TOP_K) -> Dict[str, Any]:
    """Full RAG pipeline: retrieve → prompt → Groq → answer."""

    retrieved = retrieve(query=query, top_k=top_k)

    if not retrieved:
        return {
            "answer": (
                "I don't have enough information in the "
                "provided documents to answer this."
            ),
            "sources": []
        }

    prompt = build_prompt(query=query, retrieved=retrieved)

    try:
        completion = groq_client.chat.completions.create(
            model=LLM_MODEL,                 # <-- openai/gpt-oss-20b
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a factual RAG assistant. "
                        "Only answer using the supplied document context."
                    )
                },
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            max_tokens=600,
        )
        answer = completion.choices[0].message.content.strip()

    except Exception as e:
        return {
            "answer": f"❌ Groq API error ({type(e).__name__}): {e}",
            "sources": retrieved,
        }

    return {"answer": answer, "sources": retrieved}

## 🧪 Step 10 — End-to-End Test
Runs a sample question ("What is the main topic of the documents?") through the full `generate_answer()` pipeline and prints the generated answer alongside the list of sources used (filename, page, similarity distance) — a quick manual check that ingestion → chunking → embedding → retrieval → generation all work together correctly.

In [11]:
test_q = "What is the main topic of the documents?"

print("=" * 70)
print("RAG TEST")
print("=" * 70)
print("\n❓ Question:")
print(test_q)

result = generate_answer(test_q, top_k=TOP_K)

print("\n💡 Answer:")
print(result["answer"])

print("\n📚 Sources Used:")
if result["sources"]:
    for i, s in enumerate(result["sources"], start=1):
        d = s.get("distance")
        d_text = f"{d:.4f}" if d is not None else "N/A"
        print(f"{i}. {s['source']} (page {s['page']}, distance={d_text})")
else:
    print("No sources retrieved.")

print("\n" + "=" * 70)

RAG TEST

❓ Question:
What is the main topic of the documents?

💡 Answer:
The documents cover several distinct subjects: digital transformation and its challenges in business [Source: BUSINESS.pdf, page 3]; ethical concerns and risks associated with artificial intelligence [Source: Artificial Intelligence.pdf, page 3]; the environmental impact of greenhouse‑gas emissions and climate change [Source: SCIENCE.pdf, page 1]; and copyright law, specifically the fair‑use doctrine [Source: LAW.pdf, page 1].

📚 Sources Used:
1. BUSINESS.pdf (page 3, distance=0.4406)
2. Artificial Intelligence.pdf (page 3, distance=0.4509)
3. BUSINESS.pdf (page 3, distance=0.4583)
4. SCIENCE.pdf (page 1, distance=0.4586)
5. LAW.pdf (page 1, distance=0.4605)



## 💻 Step 11 — Command-Line Interface
`cli_loop()` implements the minimum required UI: an interactive terminal loop that keeps prompting for questions, calls `generate_answer()` on each one, and prints the answer plus its sources — until the user types `exit`/`quit`/`q` or interrupts it. Left commented out (`# cli_loop()`) so it doesn't block notebook execution; uncomment to run it interactively.

In [12]:
def cli_loop():
    print("\n" + "=" * 60)
    print("🤖 RAG Document Q&A Bot (type 'exit' to quit)")
    print("=" * 60)
    while True:
        try:
            q = input("\n❓ Your question: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Goodbye!")
            break
        if not q:
            continue
        if q.lower() in {"exit", "quit", "q"}:
            print("👋 Goodbye!")
            break

        result = generate_answer(q)
        print("\n💡 Answer:\n" + result["answer"])
        print("\n📚 Sources:")
        for s in result["sources"]:
            print(f"  - {s['source']} (page {s['page']})")


# cli_loop()   # uncomment to run

## 🖥️ Step 12 — Bonus Web-Style UI (ipywidgets, two tabs)
Builds an interactive, notebook-native UI with `ipywidgets` as the bonus interface required by the assignment:
- `_render_result()` — shared helper that calls the Groq LLM on a built prompt and renders the answer plus every retrieved source chunk as Markdown
- **Tab 1 — Main RAG**: a text box, a Top-K slider, and an Ask/Clear button pair that query the full indexed document collection
- **Tab 2 — PDF Agent**: lets you upload a *new* PDF (or pick one from `data/`), index it **in-memory only** (isolated from the main Chroma collection), and ask questions scoped to just that document

The final lines assemble both tabs into a `widgets.Tab` and display it.

In [13]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output


# ============================================================
# SHARED RENDER HELPER
# ============================================================
def _render_result(query: str, retrieved: list, output_widget):
    if not retrieved:
        with output_widget:
            display(Markdown("⚠️ No relevant chunks found."))
        return

    prompt = build_prompt(query, retrieved)
    try:
        completion = groq_client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system",
                 "content": "You are a factual RAG assistant that only answers from provided context."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.1,
            max_tokens=600,
        )
        answer = completion.choices[0].message.content.strip()
    except Exception as e:
        answer = f"❌ Groq error: {e}"

    with output_widget:
        display(Markdown("### 💡 Answer"))
        display(Markdown(answer))
        display(Markdown("### 📚 Retrieved Source Chunks"))
        for i, s in enumerate(retrieved, start=1):
            d = s.get("distance")
            d_text = f"{d:.4f}" if isinstance(d, (int, float)) else "N/A"
            display(Markdown(
                f"**[{i}] {s['source']} — page {s['page']}** "
                f"_(distance: {d_text})_\n\n> {s['text'][:400]}...\n"
            ))


# ============================================================
# TAB 1 — MAIN RAG
# ============================================================
query_box = widgets.Textarea(
    value="",
    placeholder="Type your question here and press Ask...",
    description="❓ Query:",
    layout=widgets.Layout(width="100%", height="80px")
)
top_k_slider = widgets.IntSlider(
    value=TOP_K, min=1, max=8, step=1,
    description="Top-K:",
    style={"description_width": "initial"}
)
ask_btn   = widgets.Button(description="🚀 Ask",   button_style="primary")
clear_btn = widgets.Button(description="🧹 Clear", button_style="warning")
output_area = widgets.Output()


def on_ask_clicked(b):
    query = query_box.value.strip()
    with output_area:
        clear_output(wait=True)
        if not query:
            print("⚠️ Please type a question.")
            return
        print(f"❓ Question: {query}\n")
        print("🔄 Retrieving context and generating answer...\n")

    try:
        retrieved = retrieve(query, top_k=top_k_slider.value)
    except Exception as e:
        with output_area:
            print(f"❌ Error: {e}")
        return

    _render_result(query, retrieved, output_area)


def on_clear_clicked(b):
    query_box.value = ""
    with output_area:
        clear_output()


ask_btn.on_click(on_ask_clicked)
clear_btn.on_click(on_clear_clicked)


# ============================================================
# TAB 2 — PDF AGENT (isolated in-memory PDF Q&A)
# ============================================================
pdf_agent_state = {"chunks": [], "embeddings": None, "loaded_name": None}

pdf_upload = widgets.FileUpload(
    accept=".pdf", multiple=False,
    description="📎 Upload PDF",
    layout=widgets.Layout(width="250px"),
)
pdf_pick_existing = widgets.Dropdown(
    options=["— select —"] + [f.name for f in Path(DATA_DIR).glob("*.pdf")],
    description="Or pick:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="350px"),
)
pdf_load_btn = widgets.Button(description="📥 Load PDF", button_style="info")
pdf_status   = widgets.HTML(value="<i>No PDF loaded yet.</i>")

pdf_query_box = widgets.Textarea(
    value="",
    placeholder="Ask something about the loaded PDF...",
    description="❓ Ask:",
    layout=widgets.Layout(width="100%", height="80px"),
)
pdf_top_k = widgets.IntSlider(
    value=4, min=1, max=8, step=1,
    description="Top-K:",
    style={"description_width": "initial"},
)
pdf_ask_btn   = widgets.Button(description="🚀 Ask PDF", button_style="primary")
pdf_clear_btn = widgets.Button(description="🧹 Clear",   button_style="warning")
pdf_output = widgets.Output()


def _load_pdf_bytes(file_bytes: bytes, filename: str):
    reader = PdfReader(io.BytesIO(file_bytes))
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = clean_text(page.extract_text() or "")
        if text.strip():
            pages.append({"text": text, "source": filename, "page": i})

    chunks_local = []
    for p in pages:
        for j, ch in enumerate(chunk_text(p["text"])):
            chunks_local.append({
                "text":     ch,
                "source":   p["source"],
                "page":     p["page"],
                "chunk_id": f"{p['source']}_p{p['page']}_c{j}",
            })

    embs = (embed_texts([c["text"] for c in chunks_local])
            if chunks_local else np.zeros((0, 384), dtype=np.float32))

    pdf_agent_state["chunks"]      = chunks_local
    pdf_agent_state["embeddings"]  = embs
    pdf_agent_state["loaded_name"] = filename
    return len(chunks_local)


def on_pdf_load(b):
    with pdf_output:
        clear_output(wait=True)

    uploaded = pdf_upload.value
    picked   = pdf_pick_existing.value

    try:
        if uploaded:
            if isinstance(uploaded, dict):
                fname   = list(uploaded.keys())[0]
                content = uploaded[fname]["content"]
                if isinstance(content, memoryview):
                    content = content.tobytes()
            else:
                fname   = uploaded[0]["name"]
                content = uploaded[0]["content"]
            n = _load_pdf_bytes(content, fname)

        elif picked and picked != "— select —":
            with open(Path(DATA_DIR) / picked, "rb") as f:
                n = _load_pdf_bytes(f.read(), picked)
        else:
            pdf_status.value = "⚠️ Upload a PDF or pick one from <code>data/</code>."
            return

        pdf_status.value = (
            f"✅ Loaded <b>{pdf_agent_state['loaded_name']}</b> — "
            f"{n} chunks indexed (in-memory, isolated)."
        )
    except Exception as e:
        pdf_status.value = f"❌ Failed to load PDF: {e}"


pdf_load_btn.on_click(on_pdf_load)


def _retrieve_in_pdf(query: str, top_k: int):
    embs        = pdf_agent_state["embeddings"]
    chunks_local = pdf_agent_state["chunks"]
    if embs is None or len(chunks_local) == 0:
        return []

    q_vec = embed_texts([query])[0]
    q_norm = q_vec / (np.linalg.norm(q_vec) + 1e-10)
    d_norm = embs  / (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-10)
    sims   = d_norm @ q_norm

    top_idx = np.argsort(-sims)[:top_k]
    results = []
    for idx in top_idx:
        c = chunks_local[int(idx)]
        results.append({
            "text":     c["text"],
            "source":   c["source"],
            "page":     c["page"],
            "distance": float(1.0 - sims[int(idx)]),
        })
    return results


def on_pdf_ask(b):
    query = pdf_query_box.value.strip()
    with pdf_output:
        clear_output(wait=True)

    if pdf_agent_state["loaded_name"] is None:
        with pdf_output:
            print("⚠️ Load a PDF first.")
        return
    if not query:
        with pdf_output:
            print("⚠️ Type a question.")
        return

    with pdf_output:
        print(f"📄 Querying PDF: {pdf_agent_state['loaded_name']}")
        print(f"❓ Question: {query}\n")

    retrieved = _retrieve_in_pdf(query, pdf_top_k.value)
    _render_result(query, retrieved, pdf_output)


def on_pdf_clear(b):
    pdf_query_box.value = ""
    with pdf_output:
        clear_output()


pdf_ask_btn.on_click(on_pdf_ask)
pdf_clear_btn.on_click(on_pdf_clear)


# ============================================================
# RENDER — TWO TABS
# ============================================================
print("=" * 60)
print("🤖 RAG Document Q&A Bot")
print("=" * 60)

tab_main = widgets.VBox([
    widgets.HTML("<b>🔍 Query the full document collection</b>"),
    query_box,
    widgets.HBox([top_k_slider, ask_btn, clear_btn]),
    output_area,
])

tab_pdf = widgets.VBox([
    widgets.HTML("<b>📄 PDF Agent — upload or pick a PDF and ask about it</b>"),
    widgets.HBox([pdf_upload, pdf_pick_existing, pdf_load_btn]),
    pdf_status,
    widgets.HTML("<hr>"),
    pdf_query_box,
    widgets.HBox([pdf_top_k, pdf_ask_btn, pdf_clear_btn]),
    pdf_output,
])

tabs = widgets.Tab(children=[tab_main, tab_pdf])
tabs.set_title(0, "🔍 Main RAG")
tabs.set_title(1, "📄 PDF Agent")
display(tabs)

🤖 RAG Document Q&A Bot


## 💾 Step 13 — Export Artifacts (Pickle)
Re-embeds all chunks and serializes everything needed to reproduce or redeploy the index — config, chunks, embeddings, chunk/document counts, and a timestamp — into a single `rag_artifacts.pkl` file via `pickle`, and prints a short summary (chunk count, document count, embedding dimension, file size, creation time) once the export is done.

In [14]:
print("🔄 Preparing artifacts for pickle export...")

all_texts      = [c["text"] for c in chunks]
all_embeddings = embed_texts(all_texts)

artifacts = {
    "config": {
        "embedding_model": EMBEDDING_MODEL,
        "llm_model":       LLM_MODEL,          # openai/gpt-oss-20b
        "chunk_size":      CHUNK_SIZE,
        "chunk_overlap":   CHUNK_OVERLAP,
        "top_k":           TOP_K,
        "collection_name": COLLECTION_NAME,
        "chroma_dir":      CHROMA_DIR,
    },
    "chunks":        chunks,
    "embeddings":    all_embeddings.tolist(),
    "num_chunks":    len(chunks),
    "num_documents": len(set(c["source"] for c in chunks)),
    "created_at":    datetime.now().isoformat(),
}

PICKLE_PATH = "rag_artifacts.pkl"
with open(PICKLE_PATH, "wb") as f:
    pickle.dump(artifacts, f, protocol=pickle.HIGHEST_PROTOCOL)

size_mb = os.path.getsize(PICKLE_PATH) / (1024 * 1024)
print(f"✅ Saved: {PICKLE_PATH}")
print(f"   • Chunks      : {artifacts['num_chunks']}")
print(f"   • Documents   : {artifacts['num_documents']}")
print(f"   • Embed dim   : {len(artifacts['embeddings'][0])}")
print(f"   • File size   : {size_mb:.2f} MB")
print(f"   • Created at  : {artifacts['created_at']}")

🔄 Preparing artifacts for pickle export...
✅ Saved: rag_artifacts.pkl
   • Chunks      : 83
   • Documents   : 5
   • Embed dim   : 384
   • File size   : 0.31 MB
   • Created at  : 2026-09-26T07:47:04.810836
